# 03 · Tokens and context

Two constraints that drive most LLM application design:
models consume and bill **tokens**, and they are **stateless** between calls.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

## Tokenization

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")  # recent OpenAI models

text = "Streaming responses improve perceived latency in chat UIs."
ids = enc.encode(text)
print(len(text), "chars ->", len(ids), "tokens")
print(" | ".join(enc.decode([i]) for i in ids))

In [ ]:
samples = {
    "english": "The quick brown fox jumps over the lazy dog.",
    "rare words": "Pneumonoultramicroscopicsilicovolcanoconiosis is a lung disease.",
    "code": "def f(x): return [i**2 for i in range(x) if i % 2]",
    "yoruba": "Ẹ kú àárọ̀, báwo ni ìdílé yín?",
}
for label, s in samples.items():
    n = len(enc.encode(s))
    print(f"{label:11} {len(s):3} chars  {n:3} tokens  {len(s) / n:.1f} chars/token")

English averages ~4 chars/token. Code, rare words and non-English text (diacritics especially) are
markedly less efficient, which matters for both cost and context budget.

In [ ]:
def estimate_cost(prompt: str, output_tokens: int, in_per_m: float, out_per_m: float) -> float:
    """USD cost given per-million-token prices."""
    return len(enc.encode(prompt)) / 1e6 * in_per_m + output_tokens / 1e6 * out_per_m


# ~2k-char page summary at illustrative prices
print(f"${estimate_cost('x ' * 1000, 300, in_per_m=0.10, out_per_m=0.40):.6f}")

## Statelessness

In [ ]:
from toolkit import Conversation, chat

PROVIDER = "ollama"

chat([{"role": "user", "content": "Hi, my name is Isiaq."}], provider=PROVIDER)
chat([{"role": "user", "content": "What's my name?"}], provider=PROVIDER)  # no prior context

In [ ]:
chat(
    [
        {"role": "user", "content": "Hi, my name is Isiaq."},
        {"role": "assistant", "content": "Nice to meet you, Isiaq!"},
        {"role": "user", "content": "What's my name?"},
    ],
    provider=PROVIDER,
)

"Memory" is the client resending history. `toolkit.Conversation` encapsulates that; history size
(and therefore cost and latency) grows linearly with every turn until it hits the context window.

In [ ]:
convo = Conversation("Keep replies short.", provider=PROVIDER)

for msg in [
    "I'm a frontend engineer working mostly with Next.js.",
    "Suggest one AI feature for a developer portfolio site.",
    "Which framework did I say I use?",
]:
    print("user:", msg)
    print("bot: ", convo.say(msg))
    history_tokens = sum(len(enc.encode(m["content"])) for m in convo.messages)
    print(f"      [history ~{history_tokens} tokens]\n")

Implications: long-running chats need a strategy (windowing, summarizing older turns, or retrieval)
rather than unbounded history.